In [18]:
!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

--2026-08-04 15:17:01--  https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.111.133, 185.199.110.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1115394 (1.1M) [text/plain]
Saving to: ‘input.txt.1’

input.txt.1         100%[===================>]   1.06M  --.-KB/s    in 0.05s   

2026-08-04 15:17:02 (20.5 MB/s) - ‘input.txt.1’ saved [1115394/1115394]



In [19]:
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

In [20]:
from pprint import pprint
pprint(text)

Streaming output truncated to the last 5000 lines.
 'Page:\n'
 "My lord, 'tis but begun.\n"
 '\n'
 'SLY:\n'
 "'Tis a very excellent piece of work, madam lady:\n"
 "would 'twere done!\n"
 '\n'
 'PETRUCHIO:\n'
 'Verona, for a while I take my leave,\n'
 'To see my friends in Padua, but of all\n'
 'My best beloved and approved friend,\n'
 'Hortensio; and I trow this is his house.\n'
 'Here, sirrah Grumio; knock, I say.\n'
 '\n'
 'GRUMIO:\n'
 'Knock, sir! whom should I knock? is there man has\n'
 'rebused your worship?\n'
 '\n'
 'PETRUCHIO:\n'
 'Villain, I say, knock me here soundly.\n'
 '\n'
 'GRUMIO:\n'
 'Knock you here, sir! why, sir, what am I, sir, that\n'
 'I should knock you here, sir?\n'
 '\n'
 'PETRUCHIO:\n'
 'Villain, I say, knock me at this gate\n'
 "And rap me well, or I'll knock your knave's pate.\n"
 '\n'
 'GRUMIO:\n'
 'My master is grown quarrelsome. I should knock\n'
 'you first,\n'
 'And then I know after who comes by the worst.\n'
 '\n'
 'PETRUCHIO:\n'
 'Will it not be?\n'

In [21]:
chars = sorted(list(set(text)))

In [22]:
vocab_size = len(chars)

In [23]:
print("".join(chars))
print(vocab_size)


 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
65


In [24]:
chartoindex = {char : i for i, char in enumerate(chars)}
indextochar = {i : char for i, char in enumerate(chars)}
def encode(s):
  return [chartoindex[ch] for ch in s]
def decode(l):
  return ''.join([indextochar[i] for i in l])


In [25]:
print(encode("hii there"))
print(decode(encode("hii there")))

[46, 47, 47, 1, 58, 46, 43, 56, 43]
hii there


In [26]:
#encoding the whole text data
import torch
data = torch.tensor(encode(text), dtype=torch.long)
print(data.shape, data.dtype)
print(data[:100])

torch.Size([1115394]) torch.int64
tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
        58, 47, 64, 43, 52, 10,  0, 37, 53, 59])


In [27]:
n = int(0.8*len(data))
train_data = data[:n]
test_data = data[n:]

In [28]:
block_size = 8
train_data[:block_size+1]

tensor([18, 47, 56, 57, 58,  1, 15, 47, 58])

In [29]:
torch.manual_seed(1337)
batch_size = 4
block_size = 8

def get_batch(split):
  data = train_data if split == 'train' else test_data
  ix = torch.randint(len(data) - block_size, (batch_size,))
  x = torch.stack([data[i:i+block_size] for i in ix])
  y = torch.stack([data[i+1:i+block_size+1] for i in ix])
  return x, y

xb, yb = get_batch('train')
